# 01 — Data & Task Overview

What does the dataset look like? How many animals, sessions, trials?
What distributions did each animal experience, and in what order?

This notebook is pure exploration — no model fitting, no inference.


## Setup


In [ ]:
%matplotlib inline
from shared_setup import *
import pandas as pd

experiment, info = load_data()
print(f"Mode: {info['mode']}")
print(f"Animals: {experiment.n_animals}")
print(f"Total sessions: {sum(a.n_sessions for a in experiment.animals.values())}")

## Cohort Summary Table

One row per animal: session counts by stage and distribution,
total trials, date range, opto status.


In [ ]:
rows = []
for aid, animal in sorted(experiment.animals.items()):
	session_table = animal.session_table
	stages = session_table['stage']
	dists = session_table['distribution']
	has_opto = session_table['session_type'] == 'opto'
	dates = session_table['date']

	rows.append({
		'animal_id': aid,
		'n_sessions': len(session_table),
		'n_trials': session_table['n_trials'].sum(),
		'n_uniform': sum(1 for d in dists if d and 'uniform' in d.lower()),
		'n_hard_a': sum(1 for d in dists if d and 'hard' in d.lower() and 'a' in d.lower()),
		'n_hard_b': sum(1 for d in dists if d and 'hard' in d.lower() and 'b' in d.lower()),
		'has_opto': has_opto.any(),
		'first_date': min(dates).isoformat(),
		'last_date': max(dates).isoformat(),
	})

cohort_df = pd.DataFrame(rows)
print(cohort_df.to_string(index=False))


## Distribution Timeline

For each animal, show the sequence of stimulus distributions across sessions.
Opto animals follow: Uniform → Hard-A (opto) → Hard-B (opto) → ... repeating.


In [ ]:
fig, ax = plt.subplots(figsize=(16, max(4, experiment.n_animals * 0.4)))

dist_colours = {
	'Uniform': PALETTE[0],
	'Hard-A': PALETTE[1],
	'Hard-B': PALETTE[2],
}

animal_ids = sorted(experiment.animals.keys())
for y, aid in enumerate(animal_ids):
	animal = experiment.get_animal(aid)
	for i, sess in enumerate(animal.sessions):
		dist = sess.distribution or 'Unknown'
		# Match distribution name flexibly
		colour = '#CCCCCC'
		for key, col in dist_colours.items():
			if key.lower().replace('-', '') in dist.lower().replace('-', '').replace('_', ''):
				colour = col
				break

		marker = 's' if hasattr(sess.trials, 'opto_on') and sess.trials.opto_on.any() else 'o'
		ax.scatter(i, y, c=colour, marker=marker, s=25, edgecolors='white', linewidth=0.3)

ax.set_yticks(range(len(animal_ids)))
ax.set_yticklabels(animal_ids, fontsize=8)
ax.set_xlabel('Session index')
ax.set_title('Distribution Timeline (squares = opto sessions)')

# Legend
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
handles = [Patch(facecolor=c, label=k) for k, c in dist_colours.items()]
handles.append(Line2D([0], [0], marker='s', color='grey', ls='', ms=8, label='Opto'))
handles.append(Line2D([0], [0], marker='o', color='grey', ls='', ms=8, label='No opto'))
ax.legend(handles=handles, loc='upper right', fontsize=8)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Animal
Pooled psychometric for each animal's expert uniform sessions.


In [ ]:
def create_masks(animal):
	session_table = animal.session_table
 
	mask_habituation = session_table['stage'].isin(['Habituation', 'Lick_To_Release', 'Three_And_Three'])
	mask_uniform_training = session_table['distribution'].str.contains('uniform', case=False, na=False) & (session_table['stage'] == 'Full_Task_Cont') & (session_table['session_type'] == 'regular')
	indeces = mask_uniform_training.index[mask_uniform_training][-5:]
	mask_uniform_training_last5 = session_table.index.isin(indeces)
	mask_uniform_masking_light = session_table['distribution'].str.contains('uniform', case=False, na=False) & (session_table['session_type'] == 'masking')
	mask_uniform_opto = session_table['distribution'].str.contains('uniform', case=False, na=False) & (session_table['session_type'] == 'opto')
	mask_uniform_washout = session_table['distribution'].str.contains('uniform', case=False, na=False) & (session_table['session_type'] == 'washout')
	mask_hard_a_no_opto = session_table['distribution'].str.contains('hard-a', case=False, na=False) & (session_table['session_type'] == 'regular')
	mask_hard_b_no_opto = session_table['distribution'].str.contains('hard-b', case=False, na=False) & (session_table['session_type'] == 'regular')
	mask_hard_ab_opto = session_table['distribution'].str.contains('hard', case=False, na=False) & (session_table['session_type'] == 'opto')
	mask_hard_ab_masking = session_table['distribution'].str.contains('hard', case=False, na=False) & (session_table['session_type'] == 'masking')
	mask_hard_a_opto = session_table['distribution'].str.contains('hard-a', case=False, na=False) & (session_table['session_type'] == 'opto')
	mask_hard_b_opto = session_table['distribution'].str.contains('hard-b', case=False, na=False) & (session_table['session_type'] == 'opto')
	mask_hard_a_masking = session_table['distribution'].str.contains('hard-a', case=False, na=False) & (session_table['session_type'] == 'masking')
	mask_hard_b_masking = session_table['distribution'].str.contains('hard-b', case=False, na=False) & (session_table['session_type'] == 'masking')
	return {
		'habituation': mask_habituation.to_numpy(),
		'uniform_training': mask_uniform_training.to_numpy(),
		'uniform_training_last5': mask_uniform_training_last5,
		'uniform_masking_light': mask_uniform_masking_light.to_numpy(),
		'uniform_opto': mask_uniform_opto.to_numpy(),
		'uniform_washout': mask_uniform_washout.to_numpy(),
		'hard_a_no_opto': mask_hard_a_no_opto.to_numpy(),
		'hard_b_no_opto': mask_hard_b_no_opto.to_numpy(),
		'hard_ab_opto': mask_hard_ab_opto.to_numpy(),
		'hard_ab_masking': mask_hard_ab_masking.to_numpy(),
		'hard_a_opto': mask_hard_a_opto.to_numpy(),
		'hard_b_opto': mask_hard_b_opto.to_numpy(),
		'hard_a_masking': mask_hard_a_masking.to_numpy(),
		'hard_b_masking': mask_hard_b_masking.to_numpy(),
	}
 
def filter_sessions(animal = None, animal_id = None, experiment = None, mask = None):
	if animal_id:
		animal = experiment.get_animal(animal_id)
	if animal is None:
		raise ValueError("Either animal or animal_id with experiment must be provided.")
	if mask is not None:
		if not mask.any():
			return []
		else:
			return animal.get_sessions(mask=mask)
	return animal.sessions


In [ ]:
# Select animal
animal_id = 'SS13'
animal = experiment.get_animal(animal_id)

# Select sessions
masks = create_masks(animal)
sessions_last_5 = filter_sessions(animal=animal, mask=masks['uniform_training_last5'])
sessions_masking = filter_sessions(animal=animal, mask=masks['uniform_masking_light'])
sessions_opto = filter_sessions(animal=animal, mask=masks['uniform_opto'])
# Filter trials
clean_sessions_last_5 = filter_trials(sessions_last_5, mask_fn = None, min_trials = 10, label = 'last 5 sessions')
clean_sessions_masking = filter_trials(sessions_masking, mask_fn = None, min_trials = 10, label = 'masking sessions')
clean_sessions_opto = filter_trials(sessions_opto,mask_fn=lambda s: build_mask(s.trials, exclude_opto=False), 
                                    min_trials = 10, label = 'opto sessions')
clean_sessions_opto_trials_non_opto = filter_trials(sessions_opto, label='non-opto trials')
clean_sessions_opto_trials_opto = filter_trials(sessions_opto, 
                                                mask_fn=lambda s: opto_mask(s.trials, delta=0),
                                                label='opto trials')

clean_sessions_opto_trials_post_opto = filter_trials(sessions_opto,
                                                     mask_fn=lambda s: opto_mask(s.trials, delta=1),
                                                     label='post-opto trials')



In [ ]:
def compute_psyc_um(animal, session_mask = None, trial_mask = None, min_trials = 10):
    sessions_masks = create_masks(animal)
    if session_mask not in sessions_masks:
        raise ValueError(f"Session mask '{session_mask}' not found. Available masks: {list(sessions_masks.keys())}")
    selected_sessions = filter_sessions(animal=animal, mask=sessions_masks[session_mask])
    if trial_mask is not None:


In [ ]:
psyc_last_5 = compute_psychometric(clean_sessions_last_5, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)
psyc_masking = compute_psychometric(clean_sessions_masking, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)
psyc_opto = compute_psychometric(clean_sessions_opto, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)
psyc_opto_trials_non_opto = compute_psychometric(clean_sessions_opto_trials_non_opto, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)
psyc_opto_trials_opto = compute_psychometric(clean_sessions_opto_trials_opto, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)
psyc_opto_trials_post_opto = compute_psychometric(clean_sessions_opto_trials_post_opto, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
ax = axes[0, 0]
plot_psychometric(psyc_last_5, ax=ax, label=f'Last 5 Uniform Training (n={psyc_last_5["n_trials"]})', color=PALETTE[0])
plot_psychometric(psyc_masking, ax=ax, label=f'Uniform Masking (n={psyc_masking["n_trials"]})', color=PALETTE[1])
ax.set_title('Uniform - Last 5 Training vs Masking', fontsize=14)
ax.legend()

ax = axes[0, 1]
plot_psychometric(psyc_masking, ax=ax, label=f'Uniform Masking (n={psyc_masking["n_trials"]})', color=PALETTE[1])
plot_psychometric(psyc_opto, ax=ax, label=f'Uniform Opto (n={psyc_opto["n_trials"]})', color=PALETTE[2])
ax.set_title('Uniform - Masking vs Opto', fontsize=14)
ax.legend()

ax = axes[0, 2]
plot_psychometric(psyc_masking, ax=ax, label=f'Uniform Masking (n={psyc_masking["n_trials"]})', color=PALETTE[1])
plot_psychometric(psyc_opto_trials_non_opto, ax=ax, label=f'Uniform Non-Opto Trials (n={psyc_opto_trials_non_opto["n_trials"]})', color=PALETTE[3])
ax.set_title('Uniform - Masking vs Non-Opto Trials', fontsize=14)
ax.legend()

ax = axes[1, 0]
plot_psychometric(psyc_opto_trials_non_opto, ax=ax, label=f'Non-Opto Trials (n={psyc_opto_trials_non_opto["n_trials"]})', color=PALETTE[3])
plot_psychometric(psyc_opto_trials_opto, ax=ax, label=f'Opto Trials (n={psyc_opto_trials_opto["n_trials"]})', color=PALETTE[4])
ax.set_title('Uniform - Opto Session: Non-Opto vs Opto Trials', fontsize=14)
ax.legend()


ax = axes[1, 1]
plot_psychometric(psyc_opto_trials_opto, ax=ax, label=f'Opto Trials (n={psyc_opto_trials_opto["n_trials"]})', color=PALETTE[4])
plot_psychometric(psyc_opto_trials_post_opto, ax=ax, label=f'Post-Opto Trials (n={psyc_opto_trials_post_opto["n_trials"]})', color=PALETTE[5])  
ax.set_title('Uniform - Opto Session: Opto vs Post-Opto Trials', fontsize=14)
ax.legend()

ax = axes[1, 2]
plot_psychometric(psyc_opto_trials_non_opto, ax=ax, label=f'Non-Opto Trials (n={psyc_opto_trials_non_opto["n_trials"]})', color=PALETTE[3])
plot_psychometric(psyc_opto_trials_post_opto, ax=ax, label=f'Post-Opto Trials (n={psyc_opto_trials_post_opto["n_trials"]})', color=PALETTE[5])  
ax.set_title('Uniform - Opto Session: Non-Opto vs Post-Opto Trials', fontsize=14)
ax.legend()

fig.suptitle(f'{animal_id} - {animal.genotype.upper()} - Uniform Sessions - Psychometric Comparison', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
um_last_5 = compute_um(clean_sessions_last_5)
um_masking = compute_um(clean_sessions_masking)
um_opto = compute_um(clean_sessions_opto)
um_opto_trials_non_opto = compute_um(clean_sessions_opto_trials_non_opto)
um_opto_trials_opto = compute_um(clean_sessions_opto_trials_opto)
um_opto_trials_post_opto = compute_um(clean_sessions_opto_trials_post_opto)


fig, axes = plt.subplots(2, 3, figsize=(18, 10))
plot_um(um_last_5, ax=axes[0, 0], title=f'Last 5 Training (n_trials = {um_last_5["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)
plot_um(um_masking, ax=axes[0, 1], title=f'Masking (n_trials = {um_masking["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)
plot_um(um_opto, ax=axes[0, 2], title=f'Opto (n_trials = {um_opto["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)
plot_um(um_opto_trials_non_opto, ax=axes[1, 0], title=f'Non-Opto Trials (n_trials = {um_opto_trials_non_opto["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)
plot_um(um_opto_trials_opto, ax=axes[1, 1], title=f'Opto Trials (n_trials = {um_opto_trials_opto["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)
plot_um(um_opto_trials_post_opto, ax=axes[1, 2], title=f'Post-Opto Trials (n_trials = {um_opto_trials_post_opto["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)
fig.suptitle(f'{animal_id} - {animal.genotype.upper()} - Uniform - Update Matrices', fontsize=16)
plt.tight_layout()

In [ ]:
# Select sessions
sessions_hard_a_opto = filter_sessions(animal=animal, mask=masks['hard_a_opto'])
sessions_hard_b_opto = filter_sessions(animal=animal, mask=masks['hard_b_opto'])

sessions_hard_a_opto = filter_sessions(animal=animal, mask=masks['hard_a'])
sessions_hard_b_opto = filter_sessions(animal=animal, mask=masks['hard_b'])

sessions_hard_a_masking = filter_sessions(animal=animal, mask=masks['hard_a_masking'])
sessions_hard_b_masking = filter_sessions(animal=animal, mask=masks['hard_b_masking'])

# Filter trials
clean_sessions_hard_a_opto = filter_trials(sessions_hard_a_opto, mask_fn=lambda s: build_mask(s.trials, exclude_opto=False), 
                                           min_trials = 10, label = 'hard_a_opto sessions')
clean_sessions_hard_a_opto_trials_non_opto = filter_trials(sessions_hard_a_opto, label='hard-a non-opto trials')
clean_sessions_hard_a_opto_trials_opto = filter_trials(sessions_hard_a_opto, 
                                                mask_fn=lambda s: opto_mask(s.trials, delta=0),
                                                label='hard-a opto trials')
clean_sessions_hard_a_opto_trials_post_opto = filter_trials(sessions_hard_a_opto,
                                                     mask_fn=lambda s: opto_mask(s.trials, delta=1),
                                                     label='hard-a post-opto trials')

clean_sessions_hard_b_opto = filter_trials(sessions_hard_b_opto, mask_fn = lambda s: build_mask(s.trials, exclude_opto=False), 
                                           min_trials = 10, label = 'hard_b opto sessions')
clean_sessions_hard_b_opto_trials_non_opto = filter_trials(sessions_hard_b_opto, label='hard-b non-opto trials')
clean_sessions_hard_b_opto_trials_opto = filter_trials(sessions_hard_b_opto, 
                                                mask_fn=lambda s: opto_mask(s.trials, delta=0),
                                                label='hard-b opto trials')
clean_sessions_hard_b_opto_trials_post_opto = filter_trials(sessions_hard_b_opto,
                                                     mask_fn=lambda s: opto_mask(s.trials, delta=1),
                                                     label='hard-b post-opto trials')


clean_sessions_hard_a_masking = filter_trials(sessions_hard_a_masking, mask_fn = None, min_trials = 10, label = 'hard-a masking sessions')
clean_sessions_hard_b_masking = filter_trials(sessions_hard_b_masking, mask_fn = None, min_trials = 10, label = 'hard-b masking sessions')

In [ ]:
psyc_hard_a_masking = compute_psychometric(clean_sessions_hard_a_masking, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)
psyc_hard_a_opto = compute_psychometric(clean_sessions_hard_a_opto, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)
psyc_hard_a_opto_trials_non_opto = compute_psychometric(clean_sessions_hard_a_opto_trials_non_opto, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)
psyc_hard_a_opto_trials_opto = compute_psychometric(clean_sessions_hard_a_opto_trials_opto, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)
psyc_hard_a_opto_trials_post_opto = compute_psychometric(clean_sessions_hard_a_opto_trials_post_opto, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)


psyc_hard_b_masking = compute_psychometric(clean_sessions_hard_b_masking, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)
psyc_hard_b_opto = compute_psychometric(clean_sessions_hard_b_opto, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)
psyc_hard_b_opto_trials_non_opto = compute_psychometric(clean_sessions_hard_b_opto_trials_non_opto, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)
psyc_hard_b_opto_trials_opto = compute_psychometric(clean_sessions_hard_b_opto_trials_opto, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)
psyc_hard_b_opto_trials_post_opto = compute_psychometric(clean_sessions_hard_b_opto_trials_post_opto, mode = 'pooled', n_bins = 8, n_bootstrap = 1000)


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 10))

ax = axes[0, 0]
plot_psychometric(psyc_hard_a_masking, ax=ax, label=f'Hard A Masking (n={psyc_hard_a_masking["n_trials"]})', 
                  color=PALETTE[0])
plot_psychometric(psyc_hard_a_opto_trials_non_opto, ax=ax, label=f'Hard A Non-Opto Trials (n={psyc_hard_a_opto_trials_non_opto["n_trials"]})', 
                  color=PALETTE[1])
ax.set_title('Hard A - Masking vs Non-Opto Trials', fontsize=14)
ax.legend()

ax = axes[1,0]
plot_psychometric(psyc_hard_a_opto_trials_non_opto, ax=ax, label=f'Hard A Non-Opto Trials (n={psyc_hard_a_opto_trials_non_opto["n_trials"]})', 
                  color=PALETTE[1])
plot_psychometric(psyc_hard_a_opto_trials_post_opto, ax=ax, label=f'Hard A Post-Opto Trials (n={psyc_hard_a_opto_trials_post_opto["n_trials"]})', 
                  color=PALETTE[3])
ax.set_title('Hard A - Non-Opto vs Post-Opto Trials', fontsize=14)
ax.legend()


ax = axes[0, 1]
plot_psychometric(psyc_hard_b_masking, ax=ax, label=f'Hard B Masking (n={psyc_hard_b_masking["n_trials"]})', 
                  color=PALETTE[4])
plot_psychometric(psyc_hard_b_opto_trials_non_opto, ax=ax, label=f'Hard B Non-Opto Trials (n={psyc_hard_b_opto_trials_non_opto["n_trials"]})', 
                  color=PALETTE[5])
ax.set_title('Hard B - Masking vs Non-Opto Trials', fontsize=14)
ax.legend()

ax = axes[1, 1]
plot_psychometric(psyc_hard_b_opto_trials_non_opto, ax=ax, label=f'Hard B Non-Opto Trials (n={psyc_hard_b_opto_trials_non_opto["n_trials"]})', 
                  color=PALETTE[5])
plot_psychometric(psyc_hard_b_opto_trials_post_opto, ax=ax, label=f'Hard B Post-Opto Trials (n={psyc_hard_b_opto_trials_post_opto["n_trials"]})', 
                  color=PALETTE[7])
ax.set_title('Hard B - Non-Opto vs Post-Opto Trials', fontsize=14)
ax.legend()

ax = axes[0, 2]
plot_psychometric(psyc_hard_a_masking, ax=ax, label=f'Hard A Masking (n={psyc_hard_a_masking["n_trials"]})', color=PALETTE[0])
plot_psychometric(psyc_hard_b_masking, ax=ax, label=f'Hard B Masking (n={psyc_hard_b_masking["n_trials"]})', color=PALETTE[4])
ax.set_title('Hard A/B - Masking Comparison', fontsize=14)
ax.legend()

ax = axes[1, 2]
plot_psychometric(psyc_hard_a_opto_trials_non_opto, ax=ax, label=f'Hard A Non-Opto Trials (n={psyc_hard_a_opto_trials_non_opto["n_trials"]})', 
                  color=PALETTE[1])
plot_psychometric(psyc_hard_b_opto_trials_non_opto, ax=ax, label=f'Hard B Non-Opto Trials (n={psyc_hard_b_opto_trials_non_opto["n_trials"]})', 
                  color=PALETTE[5])
ax.set_title('Hard A/B - Non-Opto vs Post-Opto Trials', fontsize=14)
ax.legend()

ax = axes[0, 3]
plot_psychometric(psyc_hard_a_opto_trials_opto, ax=ax, label=f'Hard A Opto Trials (n={psyc_hard_a_opto_trials_opto["n_trials"]})', 
                  color=PALETTE[2])
plot_psychometric(psyc_hard_b_opto_trials_opto, ax=ax, label=f'Hard B Opto Trials (n={psyc_hard_b_opto_trials_opto["n_trials"]})', 
                  color=PALETTE[6])
ax.set_title('Hard A/B - Opto Trials Comparison', fontsize=14)
ax.legend()

ax = axes[1, 3]
plot_psychometric(psyc_hard_a_opto_trials_post_opto, ax=ax, label=f'Hard A Post-Opto Trials (n={psyc_hard_a_opto_trials_post_opto["n_trials"]})', 
                  color=PALETTE[3])
plot_psychometric(psyc_hard_b_opto_trials_post_opto, ax=ax, label=f'Hard B Post-Opto Trials (n={psyc_hard_b_opto_trials_post_opto["n_trials"]})', 
                  color=PALETTE[7])
ax.set_title('Hard A/B - Post-Opto Trials Comparison', fontsize=14)
ax.legend()

fig.suptitle(f'{animal_id} - {animal.genotype.upper()} - Hard A/B - Psychometric Comparison', fontsize=16)
plt.tight_layout()

plt.show()

In [ ]:
um_hard_a_masking = compute_um(clean_sessions_hard_a_masking)
um_hard_a_opto = compute_um(clean_sessions_hard_a_opto)
um_hard_a_opto_trials_non_opto = compute_um(clean_sessions_hard_a_opto_trials_non_opto)
um_hard_a_opto_trials_opto = compute_um(clean_sessions_hard_a_opto_trials_opto)
um_hard_a_opto_trials_post_opto = compute_um(clean_sessions_hard_a_opto_trials_post_opto)


um_hard_b_masking = compute_um(clean_sessions_hard_b_masking)
um_hard_b_opto = compute_um(clean_sessions_hard_b_opto)
um_hard_b_opto_trials_non_opto = compute_um(clean_sessions_hard_b_opto_trials_non_opto)
um_hard_b_opto_trials_opto = compute_um(clean_sessions_hard_b_opto_trials_opto)
um_hard_b_opto_trials_post_opto = compute_um(clean_sessions_hard_b_opto_trials_post_opto)


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 10))

plot_um(um_hard_a_masking, ax=axes[0, 0], title=f'Hard A Masking (n={um_hard_a_masking["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)
plot_um(um_hard_a_opto, ax=axes[0, 1], title=f'Hard A Opto Session (n={um_hard_a_opto["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)
plot_um(um_hard_a_opto_trials_non_opto, ax=axes[0, 2], title=f'Hard A Non-Opto Trials (n={um_hard_a_opto_trials_non_opto["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)
plot_um(um_hard_a_opto_trials_opto, ax=axes[0, 3], title=f'Hard A Opto Trials (n={um_hard_a_opto_trials_opto["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)
plot_um(um_hard_a_opto_trials_post_opto, ax=axes[0, 4], title=f'Hard A Post-Opto Trials (n={um_hard_a_opto_trials_post_opto["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)

plot_um(um_hard_b_masking, ax=axes[1, 0], title=f'Hard B Masking (n={um_hard_b_masking["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)
plot_um(um_hard_b_opto, ax=axes[1, 1], title=f'Hard B Opto Session (n={um_hard_b_opto["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)
plot_um(um_hard_b_opto_trials_non_opto, ax=axes[1, 2], title=f'Hard B Non-Opto Trials (n={um_hard_b_opto_trials_non_opto["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)
plot_um(um_hard_b_opto_trials_opto, ax=axes[1, 3], title=f'Hard B Opto Trials (n={um_hard_b_opto_trials_opto["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)
plot_um(um_hard_b_opto_trials_post_opto, ax=axes[1, 4], title=f'Hard B Post-Opto Trials (n={um_hard_b_opto_trials_post_opto["n_trials"]})', vmin=-0.35, vmax=0.35, colorbar = False)

fig.suptitle(f'{animal_id} - {animal.genotype.upper()} - Update Matrices', fontsize=16)
plt.tight_layout()
plt.show()